 Import Libraries

In [ ]:
import requests
import pandas as pd
import time

print("Ready")

Ready


The Fetcher Function

In [ ]:
def fetch_open_library(subject, limit=100, offset=0):
    url = "https://openlibrary.org/subjects/{}.json".format(
        subject.replace(" ", "_").lower()
    )
    params = {"limit": limit, "offset": offset}
    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code != 200:
            return []
        return response.json().get("works", [])
    except:
        return []

The Parser Function

In [ ]:
def parse_work(work, subject):
    # Authors
    authors = work.get("authors", [])
    author_names = ", ".join([a.get("name", "") for a in authors])

    # First publish year
    first_publish_year = work.get("first_publish_year", None)

    # Cover
    cover_id = work.get("cover_id", None)
    has_cover = 1 if cover_id else 0

    # Subjects list length = how many tags the book has
    subjects = work.get("subject", [])
    subject_count = len(subjects)

    # Editions count
    edition_count = work.get("edition_count", 0)

    # Availability
    ia = work.get("ia", [])
    is_available_online = 1 if ia else 0

    return {
        "title":               work.get("title", None),
        "authors":             author_names,
        "first_publish_year":  first_publish_year,
        "edition_count":       edition_count,
        "subject_count":       subject_count,
        "has_cover":           has_cover,
        "is_available_online": is_available_online,
        "genre":               subject,
        "cover_id":            cover_id,
        "ol_key":              work.get("key", None),
    }

 Define All 35 Genres

In [ ]:
subjects = [
    "fiction", "mystery", "science_fiction", "romance", "fantasy",
    "historical_fiction", "thriller", "horror", "adventure", "biography",
    "self_help", "psychology", "philosophy", "history", "true_crime",
    "young_adult", "children", "cooking", "travel", "business",
    "health", "science", "technology", "poetry", "religion",
    "politics", "art", "education", "sports", "environment",
    "classic_literature", "dystopia", "mythology", "humor", "drama"
]

print(f"Total subjects: {len(subjects)}")
print(f"Each subject: 5 pages × 100 books = 500 books max")
print(f"Expected total: {len(subjects) * 500} rows before dedup")

Total subjects: 35
Each subject: 5 pages × 100 books = 500 books max
Expected total: 17500 rows before dedup


The Main Collection Loop

In [ ]:
all_books = []

for subject in subjects:
    subject_count = 0
    print(f"Fetching: '{subject}'", end=" ")

    for offset in range(0, 500, 100):   # 5 pages of 100 each
        works = fetch_open_library(subject, limit=100, offset=offset)
        if not works:
            break
        for work in works:
            parsed = parse_work(work, subject)
            all_books.append(parsed)
            subject_count += 1
        time.sleep(0.5)

    print(f"→ {subject_count} books")

print(f"\nTotal collected: {len(all_books)}")

Fetching: 'fiction' → 500 books
Fetching: 'mystery' → 500 books
Fetching: 'science_fiction' → 500 books
Fetching: 'romance' → 500 books
Fetching: 'fantasy' → 500 books
Fetching: 'historical_fiction' → 500 books
Fetching: 'thriller' → 500 books
Fetching: 'horror' → 500 books
Fetching: 'adventure' → 500 books
Fetching: 'biography' → 500 books
Fetching: 'self_help' → 500 books
Fetching: 'psychology' → 500 books
Fetching: 'philosophy' → 500 books
Fetching: 'history' → 500 books
Fetching: 'true_crime' → 500 books
Fetching: 'young_adult' → 500 books
Fetching: 'children' → 500 books
Fetching: 'cooking' → 500 books
Fetching: 'travel' → 500 books
Fetching: 'business' → 500 books
Fetching: 'health' → 500 books
Fetching: 'science' → 500 books
Fetching: 'technology' → 500 books
Fetching: 'poetry' → 500 books
Fetching: 'religion' → 500 books
Fetching: 'politics' → 500 books
Fetching: 'art' → 500 books
Fetching: 'education' → 500 books
Fetching: 'sports' → 500 books
Fetching: 'environment' → 500 boo

Convert to DataFrame & Remove Duplicates

In [ ]:
df = pd.DataFrame(all_books)
print("Before dedup:", df.shape)

df.drop_duplicates(subset=["title", "authors"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("After dedup:", df.shape)
df.head()

Before dedup: (17118, 10)
After dedup: (14021, 10)


,title,authors,first_publish_year,edition_count,subject_count,has_cover,is_available_online,genre,cover_id,ol_key
0,Pride and Prejudice,Jane Austen,1813.0,4038,95,1,1,fiction,14348537.0,/works/OL66554W
1,Alice's Adventures in Wonderland,Lewis Carroll,1865.0,3546,176,1,1,fiction,10527843.0,/works/OL138052W
2,A Christmas Carol,Charles Dickens,1843.0,3198,186,1,1,fiction,12875748.0,/works/OL32466W
3,The Picture of Dorian Gray,Oscar Wilde,1890.0,3012,66,1,1,fiction,14314858.0,/works/OL8193416W
4,Wuthering Heights,Emily Brontë,1846.0,2886,119,1,1,fiction,12818862.0,/works/OL21177W


Fetch Ratings for Every Book

In [ ]:
# Fetch extra details (ratings) for each book using its ol_key
# We do this for a sample to keep it fast

def fetch_ratings(ol_key):
    """ol_key looks like /works/OL45804W"""
    try:
        url = f"https://openlibrary.org{ol_key}/ratings.json"
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None, None
        data = r.json()
        summary = data.get("summary", {})
        return summary.get("average", None), summary.get("count", None)
    except:
        return None, None

# Fetch ratings for all books (will take ~15-20 mins for 3000+ books)
# If too slow, reduce to df.head(3000)

ratings_avg = []
ratings_count = []

total = len(df)
for i, key in enumerate(df["ol_key"]):
    avg, count = fetch_ratings(key)
    ratings_avg.append(avg)
    ratings_count.append(count)
    if i % 200 == 0:
        print(f"Progress: {i}/{total}")
    time.sleep(0.1)

df["avg_rating"] = ratings_avg
df["ratings_count"] = ratings_count

print("Ratings fetched!")
print(df[["title", "avg_rating", "ratings_count"]].dropna().head(10))

Progress: 0/14021
Progress: 200/14021
Progress: 400/14021
Progress: 600/14021
Progress: 800/14021
Progress: 1000/14021
Progress: 1200/14021
Progress: 1400/14021
Progress: 1600/14021
Progress: 1800/14021
Progress: 2000/14021
Progress: 2200/14021
Progress: 2400/14021
Progress: 2600/14021
Progress: 2800/14021
Progress: 3000/14021
Progress: 3200/14021
Progress: 3400/14021
Progress: 3600/14021
Progress: 3800/14021
Progress: 4000/14021
Progress: 4200/14021
Progress: 4400/14021
Progress: 4600/14021
Progress: 4800/14021
Progress: 5000/14021
Progress: 5200/14021
Progress: 5400/14021
Progress: 5600/14021
Progress: 5800/14021
Progress: 6000/14021
Progress: 6200/14021
Progress: 6400/14021
Progress: 6600/14021
Progress: 6800/14021
Progress: 7000/14021
Progress: 7200/14021
Progress: 7400/14021
Progress: 7600/14021
Progress: 7800/14021
Progress: 8000/14021
Progress: 8200/14021
Progress: 8400/14021
Progress: 8600/14021
Progress: 8800/14021
Progress: 9000/14021
Progress: 9200/14021
Progress: 9400/14021

 First Version of Popularity Score

In [ ]:
# Book age
df["book_age"] = 2025 - df["first_publish_year"]
df["book_age"] = df["book_age"].clip(lower=0)

# Popularity score (for ALL rows, using edition_count + ratings)
df["ratings_count_filled"] = df["ratings_count"].fillna(0)
df["avg_rating_filled"] = df["avg_rating"].fillna(df["avg_rating"].median())

df["popularity_score"] = (
    df["avg_rating_filled"] * 0.4 +
    df["ratings_count_filled"].apply(lambda x: min(x, 500) / 500) * 0.3 +
    df["edition_count"].apply(lambda x: min(x, 50) / 50) * 0.3
) * 10

# Binary target
threshold = df["popularity_score"].quantile(0.70)
df["is_popular"] = (df["popularity_score"] >= threshold).astype(int)

print("Columns:", df.columns.tolist())
print("Total columns:", len(df.columns))
print("Total rows:", len(df))
print(f"\nPopular: {df['is_popular'].sum()} | Not popular: {(df['is_popular']==0).sum()}")

Columns: ['title', 'authors', 'first_publish_year', 'edition_count', 'subject_count', 'has_cover', 'is_available_online', 'genre', 'cover_id', 'ol_key', 'avg_rating', 'ratings_count', 'book_age', 'ratings_count_filled', 'avg_rating_filled', 'popularity_score', 'is_popular']
Total columns: 17
Total rows: 14021

Popular: 4207 | Not popular: 9814


In [ ]:
df.to_csv("books_dataset.csv", index=False)
print("Saved: books_dataset.csv")
print(f"Final shape: {df.shape}")

Saved: books_dataset.csv
Final shape: (14021, 17)
